# 020. LSTM/GRU input/output shape

- return_sequences = False, True 일 때의 output 비교

- return_state = False, True 일 때의 internal state output 비교

- Bidirectional LSTM/GRU 의 output 비교

In [1]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, GRU
import numpy as np

T = 4   #Time Steps
D = 3   #features
U = 6   #LSTM units

X = np.random.randn(2, T, D)
print(X.shape)
X

(2, 4, 3)


array([[[-0.72890655,  0.45588127,  0.98378465],
        [-1.35421959, -0.02045508, -0.10095934],
        [-1.12483292, -0.6667466 , -1.0818938 ],
        [-1.37272874, -0.30730049, -2.2618409 ]],

       [[-0.34959643, -2.30508725, -1.25250416],
        [ 1.06792502, -0.10479824,  0.61286377],
        [ 0.66995581,  0.53937334, -0.57564617],
        [-0.17258284,  1.09729823,  0.45854578]]])

# LSTM

## return_sequences

- False (default) - last time step 의 output 만 반환
- True - 모든 timestep 의 output 을 모두 반환

In [2]:
def lstm(return_sequences=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)

    return model.predict(X)

print("---- return_sequences=False ----> last timestep 의 output 만 반환")
lstm_out = lstm(return_sequences=False)
print(lstm_out.shape)
print(lstm_out)

print("\n---- return_sequences=True ----> 모든 timestep 별 output 출력")
lstm_out = lstm(return_sequences=True)
print(lstm_out.shape)
print(lstm_out)

---- return_sequences=False ----> last timestep 의 output 만 반환
(2, 6)
[[ 0.13467756  0.13338618 -0.24769841  0.14872564 -0.13001308 -0.14605014]
 [-0.09482373  0.00246053 -0.05557596 -0.00089225  0.14314829  0.02950758]]

---- return_sequences=True ----> 모든 timestep 별 output 출력
(2, 4, 6)
[[[ 0.159671   -0.02191334  0.05274255 -0.00693318 -0.02800917
   -0.09067427]
  [ 0.20439515  0.02113756 -0.02526224 -0.04401344  0.01513339
    0.00914714]
  [ 0.07368139  0.04191491 -0.19777156 -0.04806467  0.08118355
    0.1558713 ]
  [-0.06998834  0.04060789 -0.36733812 -0.11352828  0.08495868
    0.24430044]]

 [[-0.18214807  0.00769593 -0.26665467  0.058242    0.14881015
    0.18253891]
  [-0.0192116  -0.0964925  -0.08981384  0.1822867   0.04285278
    0.09163851]
  [-0.01420432 -0.04838689 -0.03776832  0.03802398 -0.02691376
    0.03091971]
  [ 0.10398434 -0.03258112  0.08216189 -0.09146117 -0.09950298
   -0.09392283]]]


## return_state

- False (default) - output 만 반환

- True - output, last step 의 hidden state, cell state (LSTM 의 경우) 반환

In [3]:
def lstm(return_state=False):
    input_ = Input(shape=(T, D))
    rnn = LSTM(U, return_state=return_state)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:
        o, h, c = model.predict(X)
        print("o :", o, o.shape)
        print("h :", h, h.shape)
        print("c :", c, c.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- return_state=False ----> outout only")       
lstm(return_state=False)
print("\n---- return_state=True ----> outout, hidden state, cell state all")      
lstm(return_state=True)

---- return_state=False ----> outout only
o : [[-0.01186179  0.04026127  0.08544029  0.39694297 -0.02880755 -0.15317132]
 [-0.0004323  -0.03262274 -0.00052595 -0.08487893  0.02228988  0.02713225]] (2, 6)

---- return_state=True ----> outout, hidden state, cell state all
o : [[-0.12162217  0.03180042 -0.00823267  0.04522992 -0.5349265  -0.2442408 ]
 [ 0.15357268  0.09699766  0.08205333  0.06352041  0.0579636  -0.02498986]] (2, 6)
h : [[-0.12162217  0.03180042 -0.00823267  0.04522992 -0.5349265  -0.2442408 ]
 [ 0.15357268  0.09699766  0.08205333  0.06352041  0.0579636  -0.02498986]] (2, 6)
c : [[-0.8417559   0.12266982 -0.03336522  0.0806072  -1.2176418  -0.46327692]
 [ 0.25289527  0.14977723  0.15736759  0.15303843  0.15276852 -0.0558732 ]] (2, 6)


# Bidirectional LSTM

- 순방향, 역방향이 concatenate 된 output 출력  

- hidden state, cell state 는 순방향, 역방향 별도 출력

In [4]:
T, D, U

(4, 3, 6)

In [7]:
def bi_lstm(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(LSTM(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h1, c1, h2, c2 = model.predict(X)
        print("o :", o, o.shape)
        print("h1 :", h1, h1.shape)
        print("c1 :", c1, c1.shape)
        print("h2 :", h2, h2.shape)
        print("c2 :", c2, c2.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- 순방향, 역방향이 concatenate 된 many-to-one output")
bi_lstm(return_sequences=False, return_state=False)
print()
print("---- 순방향, 역방향이 concatenate 된 many-to-many output")
bi_lstm(return_sequences=True)
print()
print("---- 순방향, 역방향이 concatenate 된 sequence-to-vector output")
bi_lstm(return_state=True)

---- 순방향, 역방향이 concatenate 된 many-to-one output
o : [[-0.07930008 -0.09378726 -0.16094047 -0.22111681  0.29199693  0.03736222
  -0.321398    0.16573282 -0.3144448  -0.18838677  0.218831    0.21211086]
 [ 0.08083054  0.18861856  0.00220273 -0.00835617 -0.08119079 -0.05587874
  -0.08875515 -0.20616521 -0.10338689  0.02127507 -0.12968044  0.05731214]] (2, 12)

---- 순방향, 역방향이 concatenate 된 many-to-many output
o : [[[ 0.00961447 -0.08219736  0.10773448 -0.10388364  0.05063041
   -0.18189907 -0.03454451 -0.01513196  0.00927377  0.19727424
    0.07502981 -0.096649  ]
  [ 0.02027682 -0.03836212  0.23360708 -0.14467949  0.11964364
   -0.27691078  0.01062224 -0.05200498 -0.10222202  0.08413479
    0.1987024  -0.21503659]
  [ 0.02928887  0.0932802   0.27054778 -0.11598813  0.11612059
   -0.22210404  0.02971049 -0.05285231 -0.10439153  0.01717995
    0.25644398 -0.29572687]
  [-0.08095841  0.33572984  0.23204936 -0.12687302  0.11356014
   -0.15688851  0.02551904 -0.02634654 -0.04475056  0.00482996

# GRU 

- cell state 가 없는 것만 LSTM 과 차이

In [10]:
def gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = GRU(U, return_state=return_state, return_sequences=return_sequences)
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    
    if return_state:    
        o, h = model.predict(X)
        print("o :", o, o.shape)
        print("h :", h, h.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)

print("---- Many-to-One output ----")
gru(return_sequences=False, return_state=False)
print()
print("---- Many-to-Many output ----")
gru(return_sequences=True)
print()
print("---- Sequence-to-Vector output ----")
gru(return_state=True)

---- Many-to-One output ----
o : [[ 0.21819581 -0.45987576 -0.5318709  -0.6181512   0.35599276 -0.52554715]
 [-0.18742354 -0.05583762 -0.10924202  0.01868088 -0.19915427 -0.19632643]] (2, 6)

---- Many-to-Many output ----
o : [[[ 0.07528712 -0.1344111  -0.12669025  0.00343956  0.31749475
    0.1280259 ]
  [ 0.12635395 -0.04037698 -0.17819305  0.2962794   0.17815337
    0.20228821]
  [ 0.18882427  0.2584169  -0.10039745  0.43991303 -0.22062314
    0.1234441 ]
  [ 0.15580046  0.43659618 -0.17392288  0.5508598  -0.48146272
    0.04232774]]

 [[ 0.30016303  0.6759433   0.2698009   0.14995599 -0.41337252
   -0.1639382 ]
  [ 0.11196789  0.12407087  0.27009946 -0.2644592  -0.17895788
   -0.17626582]
  [-0.15002854 -0.0163752   0.13560101 -0.37960786 -0.10622253
   -0.16537076]
  [-0.2220613  -0.21010077 -0.0952425  -0.4449068   0.3029263
   -0.00148693]]] (2, 4, 6)

---- Sequence-to-Vector output ----
o : [[-0.31487757  0.5865503  -0.37887734  0.2919348  -0.45926255  0.42837942]
 [-0.0592451 

# Bidirectional GRU

- cell state 가 없는 것 외에 LSTM 과 동일

In [13]:
def bi_gru(return_sequences=False, return_state=False):
    input_ = Input(shape=(T, D))
    rnn = Bidirectional(GRU(U, return_state=return_state, return_sequences=return_sequences))
    x = rnn(input_)
    
    model = Model(inputs=input_, outputs=x)
    if return_state:    
        o, h1, h2 = model.predict(X)
        print("o :", o, o.shape)
        print("h1 :", h1, h1.shape)
        print("h2 :", h2, h2.shape)
    else:
        o = model.predict(X)
        print("o :", o, o.shape)
        
print("---- 순방향, 역방향이 concatenate 된 many-to-one output")
bi_gru(return_sequences=False, return_state=False)
print()
print("---- 순방향, 역방향이 concatenate 된 many-to-many output")
bi_gru(return_sequences=True)
print()
print("---- 순방향, 역방향이 concatenate 된 sequence-to-vector output")
bi_gru(return_state=True)

---- 순방향, 역방향이 concatenate 된 many-to-one output
o : [[ 0.69605327  0.21744856  0.39079615  0.26230216 -0.31045023  0.00772971
  -0.3091525   0.16694814  0.23727612  0.4729073  -0.30768037 -0.5020814 ]
 [-0.11018176 -0.13992144 -0.11164168 -0.06306054  0.09126578  0.29676044
   0.2265541   0.26593298 -0.0624781  -0.07594876 -0.11753913  0.56285846]] (2, 12)

---- 순방향, 역방향이 concatenate 된 many-to-many output
o : [[[-0.297287   -0.08380557 -0.14838107 -0.13356143 -0.20620826
    0.29551685 -0.04333649 -0.479428    0.46211752 -0.08938807
    0.15897758  0.2443601 ]
  [-0.19584058 -0.15522134  0.14717042 -0.23344047 -0.26888373
    0.40247053 -0.08339459 -0.6029763   0.5299173  -0.00578336
    0.22051534  0.06828791]
  [ 0.20838095 -0.11554962  0.38286132 -0.17480716 -0.07221165
    0.19472629 -0.10603339 -0.5740411   0.4399951   0.10764726
    0.16595197 -0.07030764]
  [ 0.63309455 -0.02374701  0.496998    0.0465308  -0.00211762
   -0.17027345 -0.12890682 -0.5167724   0.31543502  0.22846451